In [1]:
from pathlib import Path

ROOT = Path(r"C:\Users\Roberto\PEARL")

matches = sorted(
    p for p in ROOT.rglob("*.ipynb")
    if "09i" in p.name.lower()
)

print("=== NOTEBOOK 09i TROVATI ===")

for p in matches:
    print(p)

print("\nTotale:", len(matches))

=== NOTEBOOK 09i TROVATI ===
C:\Users\Roberto\PEARL\09i_MOL00583_EGFR_MMGBSA_Endpoint_Analysis.ipynb

Totale: 1


In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\Roberto\PEARL")

SOURCE_ROOT = PROJECT_ROOT / "MOL00484_09l_WINDOWS_1ns"

RUN_09L = (
    SOURCE_ROOT
    / "production_runs"
    / "20261027T171745_698045Z_seed20261027"
)

FILES = {
    "physical_system":
        SOURCE_ROOT / "states" / "physical_system.xml",

    "atom_map":
        SOURCE_ROOT / "tables" / "solute_atom_map.csv",

    "bonds":
        SOURCE_ROOT / "tables" / "solute_bonds.json",

    "ligand_openff":
        SOURCE_ROOT / "inputs" / "ligand_openff.json",

    "solvated_pdb":
        SOURCE_ROOT / "structures" / "solvated.pdb",

    "production_solute":
        RUN_09L / "trajectories" / "production_solute.dcd",

    "A26b_summary":
        RUN_09L / "tables" /
        "09l_A26b_EGF_interface_trajectory_relevance.csv",

    "A26b_occupancy":
        RUN_09L / "tables" /
        "09l_A26b_EGF_interface_residue_occupancy.csv",

    "production_log":
        RUN_09L / "tables" / "production.csv",

    "provenance":
        RUN_09L / "provenance.json",
}

print("=== 09m SOURCE PREFLIGHT ===")

all_ok = True

for name, path in FILES.items():
    exists = path.is_file()
    all_ok &= exists

    if exists:
        size = path.stat().st_size / 1024**2
        print(f"{name:20s}: OK   {size:10.2f} MB")
    else:
        print(f"{name:20s}: MISSING")
        
print("\nSource run:", RUN_09L)
print("All required inputs present:", all_ok)

assert all_ok

atommap = pd.read_csv(FILES["atom_map"])

R = atommap.loc[
    atommap["component"] == "EGFR",
    "index"
].to_numpy(dtype=int)

L = atommap.loc[
    atommap["component"] == "MOL00484",
    "index"
].to_numpy(dtype=int)

S = np.sort(np.concatenate([R, L]))

print("\n=== COMPONENTS ===")
print("EGFR atoms     :", len(R))
print("MOL00484 atoms:", len(L))
print("Total solute   :", len(S))
print("Index range    :", S.min(), "->", S.max())

assert len(R) == 9286
assert len(L) == 46
assert len(S) == 9332
assert np.array_equal(S, np.arange(9332))

print("\n09m SOURCE PREFLIGHT: OK")

=== 09m SOURCE PREFLIGHT ===
physical_system     : OK        66.11 MB
atom_map            : OK         0.27 MB
bonds               : OK         0.25 MB
ligand_openff       : OK         0.01 MB
solvated_pdb        : OK        24.03 MB
production_solute   : OK        53.44 MB
A26b_summary        : OK         0.00 MB
A26b_occupancy      : OK         0.00 MB
production_log      : OK         0.07 MB
provenance          : OK         0.00 MB

Source run: C:\Users\Roberto\PEARL\MOL00484_09l_WINDOWS_1ns\production_runs\20261027T171745_698045Z_seed20261027
All required inputs present: True

=== COMPONENTS ===
EGFR atoms     : 9286
MOL00484 atoms: 46
Total solute   : 9332
Index range    : 0 -> 9331

09m SOURCE PREFLIGHT: OK


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Roberto\PEARL")

SOURCE_ROOT = PROJECT_ROOT / "MOL00484_09l_WINDOWS_1ns"

RUN_09L = (
    SOURCE_ROOT
    / "production_runs"
    / "20261027T171745_698045Z_seed20261027"
)

FILES = {
    "physical_system":
        SOURCE_ROOT / "states" / "physical_system.xml",

    "atom_map":
        SOURCE_ROOT / "tables" / "solute_atom_map.csv",

    "bonds":
        SOURCE_ROOT / "tables" / "solute_bonds.json",

    "ligand_openff":
        SOURCE_ROOT / "inputs" / "ligand_openff.json",

    "solvated_pdb":
        SOURCE_ROOT / "structures" / "solvated.pdb",

    "production_solute":
        RUN_09L / "trajectories" / "production_solute.dcd",

    "A26b_summary":
        RUN_09L / "tables" /
        "09l_A26b_EGF_interface_trajectory_relevance.csv",

    "A26b_occupancy":
        RUN_09L / "tables" /
        "09l_A26b_EGF_interface_residue_occupancy.csv",

    "production_log":
        RUN_09L / "tables" / "production.csv",

    "provenance":
        RUN_09L / "provenance.json",
}

print("FILES ricostruito:", len(FILES), "percorsi")

for name, path in FILES.items():
    print(f"{name:20s}: {path.is_file()}")

assert all(path.is_file() for path in FILES.values())

print("\n09m PATHS: OK")

FILES ricostruito: 10 percorsi
physical_system     : True
atom_map            : True
bonds               : True
ligand_openff       : True
solvated_pdb        : True
production_solute   : True
A26b_summary        : True
A26b_occupancy      : True
production_log      : True
provenance          : True

09m PATHS: OK


In [3]:
import sys
import json
import hashlib
import inspect
import importlib.metadata

import numpy as np
import pandas as pd
import openmm as mm
from openmm import app, unit
from openmm.app.internal import customgbforces
from openmm.app.internal.customgbforces import GBSAGBn2Force
import mdtraj as mt

# ============================================================
# 09m configuration — smoke test
# ============================================================

CANDIDATE = "MOL00484"
TIER = "TIER_2_ORTHOGONAL_FOLLOWUP_LEAD"

FAST_TEST_MODE = True
N_SNAPSHOTS = 3

DISCARD_FRACTION = 0.20

SOLUTE_DIELECTRIC = 1.0
SOLVENT_DIELECTRIC = 78.5
KAPPA_NM_INV = 0.0

# Endpoint analysis on CPU, as in historical 09i
PLATFORM = "CPU"
CPU_THREADS = 4

FRAME_PS = 2.0

def require(ok, message):
    if not ok:
        raise RuntimeError(message)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# ============================================================
# Load physical system and atom map
# ============================================================

physical = mm.XmlSerializer.deserialize(
    FILES["physical_system"].read_text()
)

atommap = pd.read_csv(
    FILES["atom_map"],
    keep_default_na=False
)

bonds = json.loads(
    FILES["bonds"].read_text()
)

R = atommap.loc[
    atommap["component"] == "EGFR",
    "index"
].to_numpy(dtype=int)

L = atommap.loc[
    atommap["component"] == "MOL00484",
    "index"
].to_numpy(dtype=int)

S = np.sort(np.concatenate([R, L]))

n = len(S)

require(n == 9332, "Numero atomi del soluto inatteso.")
require(len(R) == 9286, "Numero atomi EGFR inatteso.")
require(len(L) == 46, "Numero atomi MOL00484 inatteso.")
require(
    np.array_equal(S, np.arange(n)),
    "Gli indici del soluto non sono contigui 0..n-1."
)

# ============================================================
# Recover solute topology from solvated PDB
# ============================================================

pdb = app.PDBFile(str(FILES["solvated_pdb"]))

top = app.Topology()

chain_map = {}
residue_map = {}
atom_objects = []

for i, atom in enumerate(pdb.topology.atoms()):
    if i >= n:
        break

    old_chain = atom.residue.chain

    if old_chain.index not in chain_map:
        chain_map[old_chain.index] = top.addChain(
            id=old_chain.id
        )

    chain = chain_map[old_chain.index]

    rkey = (
        old_chain.index,
        atom.residue.index
    )

    if rkey not in residue_map:
        residue_map[rkey] = top.addResidue(
            atom.residue.name,
            chain,
            id=atom.residue.id,
            insertionCode=atom.residue.insertionCode
        )

    new_atom = top.addAtom(
        atom.name,
        atom.element,
        residue_map[rkey],
        id=atom.id
    )

    atom_objects.append(new_atom)

require(
    len(atom_objects) == n,
    "Ricostruzione topologia soluto incompleta."
)

# Add bonds from authoritative 09l bond table
for a, b in bonds:
    top.addBond(
        atom_objects[int(a)],
        atom_objects[int(b)]
    )

print("=== 09m ENVIRONMENT ===")
print("Python :", sys.version.split()[0])
print("OpenMM :", importlib.metadata.version("openmm"))
print("MDTraj :", importlib.metadata.version("mdtraj"))

print("\n=== SOLUTE TOPOLOGY ===")
print("Atoms       :", top.getNumAtoms())
print("Residues    :", top.getNumResidues())
print("Chains      :", top.getNumChains())
print("Bonds       :", sum(1 for _ in top.bonds()))

# ============================================================
# Load lightweight 09l solute trajectory
# ============================================================

mdtop = mt.Topology.from_openmm(top)

trajectory = mt.load_dcd(
    str(FILES["production_solute"]),
    top=mdtop
)

print("\n=== SOURCE TRAJECTORY ===")
print("Frames :", trajectory.n_frames)
print("Atoms  :", trajectory.n_atoms)

require(
    trajectory.n_frames == 500,
    "La production 09l deve contenere 500 frame."
)

require(
    trajectory.n_atoms == n,
    "Traiettoria/topologia incompatibili."
)

require(
    np.isfinite(trajectory.xyz).all(),
    "Coordinate non finite."
)

# ============================================================
# Bond continuity check
# ============================================================

bond_array = np.asarray(bonds, dtype=int)

lengths = mt.compute_distances(
    trajectory,
    bond_array,
    periodic=False
)

max_bond_nm = float(lengths.max())

print("\nMaximum bond length:", max_bond_nm, "nm")

require(
    max_bond_nm < 0.30,
    "Legami >3 Å: possibile problema di continuità PBC."
)

# ============================================================
# Select 3 smoke-test snapshots after 20% discard
# ============================================================

start = int(
    trajectory.n_frames * DISCARD_FRACTION
)

frames = np.linspace(
    start,
    trajectory.n_frames - 1,
    N_SNAPSHOTS,
    dtype=int
)

require(
    len(np.unique(frames)) == N_SNAPSHOTS,
    "Frame duplicati."
)

frame_table = pd.DataFrame({
    "frame_index_0": frames,
    "time_ps": (frames + 1) * FRAME_PS
})

print("\n=== 3 TEST SNAPSHOTS ===")
print(frame_table.to_string(index=False))

print("\nGB implementation:")
print(inspect.getfile(customgbforces))
print("SHA256:", sha(inspect.getfile(customgbforces)))

print("\n09m TOPOLOGY + TRAJECTORY PREFLIGHT: OK")

=== 09m ENVIRONMENT ===
Python : 3.11.16
OpenMM : 8.6.1
MDTraj : 1.11.1

=== SOLUTE TOPOLOGY ===
Atoms       : 9332
Residues    : 615
Chains      : 2
Bonds       : 9451

=== SOURCE TRAJECTORY ===
Frames : 500
Atoms  : 9332

Maximum bond length: 0.22028952836990356 nm

=== 3 TEST SNAPSHOTS ===
 frame_index_0  time_ps
           100    202.0
           299    600.0
           499   1000.0

GB implementation:
C:\Users\Roberto\anaconda3\envs\pearl-gpu\Lib\site-packages\openmm\app\internal\customgbforces.py
SHA256: f96a49a9e9ef0528e937481b1c1331b43ad03e6ec3018b478c642b301f07391a

09m TOPOLOGY + TRAJECTORY PREFLIGHT: OK


In [4]:
# ============================================================
# 09m — Extract authoritative MM parameters
#       and build GBn2 parameters
# ============================================================

# Find the NonbondedForce in the authoritative physical system
nbforces = [
    f for f in physical.getForces()
    if isinstance(f, mm.NonbondedForce)
]

require(
    len(nbforces) == 1,
    f"Atteso 1 NonbondedForce, trovati {len(nbforces)}."
)

nbforce = nbforces[0]

require(
    nbforce.getNumParticles() >= n,
    "NonbondedForce contiene meno particelle del soluto."
)

# ------------------------------------------------------------
# Extract charge, sigma, epsilon for the 9332 solute atoms
# ------------------------------------------------------------

q = np.zeros(n, dtype=float)
sigma = np.zeros(n, dtype=float)
epsilon = np.zeros(n, dtype=float)

for i in range(n):
    qi, si, ei = nbforce.getParticleParameters(i)

    q[i] = qi.value_in_unit(unit.elementary_charge)
    sigma[i] = si.value_in_unit(unit.nanometer)
    epsilon[i] = ei.value_in_unit(
        unit.kilojoule_per_mole
    )

require(np.isfinite(q).all(), "Cariche non finite.")
require(np.isfinite(sigma).all(), "Sigma non finite.")
require(np.isfinite(epsilon).all(), "Epsilon non finite.")

require(
    (sigma > 0).all(),
    "Trovati sigma <= 0."
)

require(
    (epsilon >= 0).all(),
    "Trovati epsilon < 0."
)

# ------------------------------------------------------------
# Basic charge diagnostics
# ------------------------------------------------------------

q_total = float(q.sum())
q_receptor = float(q[R].sum())
q_ligand = float(q[L].sum())

print("=== AUTHORITATIVE MM PARAMETERS ===")
print("Nonbonded particles :", nbforce.getNumParticles())
print("Solute atoms        :", n)
print()
print(f"Total solute charge : {q_total:.6f} e")
print(f"EGFR charge         : {q_receptor:.6f} e")
print(f"MOL00484 charge     : {q_ligand:.6f} e")

print()
print(
    "Sigma range        :",
    float(sigma.min()),
    "->",
    float(sigma.max()),
    "nm"
)

print(
    "Epsilon range      :",
    float(epsilon.min()),
    "->",
    float(epsilon.max()),
    "kJ/mol"
)

# ------------------------------------------------------------
# Safety check: no receptor-ligand covalent exceptions
# ------------------------------------------------------------

R_set = set(map(int, R))
L_set = set(map(int, L))

cross_exceptions = []

for j in range(nbforce.getNumExceptions()):
    a, b, chargeProd, sig, eps = (
        nbforce.getExceptionParameters(j)
    )

    a = int(a)
    b = int(b)

    if (
        (a in R_set and b in L_set)
        or
        (a in L_set and b in R_set)
    ):
        cross_exceptions.append((a, b))

print()
print("Total exceptions    :", nbforce.getNumExceptions())
print("R-L exceptions      :", len(cross_exceptions))

require(
    len(cross_exceptions) == 0,
    "Esistono eccezioni NonbondedForce tra EGFR e MOL00484."
)

# ------------------------------------------------------------
# GBn2 standard parameters from complete solute topology
# ------------------------------------------------------------

gbparams = np.asarray(
    GBSAGBn2Force.getStandardParameters(top),
    dtype=float
)

require(
    gbparams.shape == (n, 5),
    f"Shape GBn2 inattesa: {gbparams.shape}"
)

require(
    np.isfinite(gbparams).all(),
    "Parametri GBn2 non finiti."
)

require(
    (gbparams[:, 0] > 0).all(),
    "Raggi GBn2 non validi."
)

print("\n=== GBn2 PARAMETERS ===")
print("Shape :", gbparams.shape)

print(
    "Radius range:",
    float(gbparams[:, 0].min()),
    "->",
    float(gbparams[:, 0].max()),
    "nm"
)

print(
    "Screen range:",
    float(gbparams[:, 1].min()),
    "->",
    float(gbparams[:, 1].max())
)

# ------------------------------------------------------------
# Prepare parameter table
# ------------------------------------------------------------

params = atommap.copy()

params["charge_e"] = q
params["sigma_nm"] = sigma
params["epsilon_kJ_mol"] = epsilon

for j, name in enumerate(
    ["radius_nm", "screen", "alpha", "beta", "gamma"]
):
    params[name] = gbparams[:, j]

print("\nParameter table rows:", len(params))

require(
    len(params) == n,
    "Tabella parametri incompleta."
)

print("\n09m MM + GBn2 PARAMETERIZATION: OK")

=== AUTHORITATIVE MM PARAMETERS ===
Nonbonded particles : 311112
Solute atoms        : 9332

Total solute charge : -4.000000 e
EGFR charge         : -4.000000 e
MOL00484 charge     : 0.000000 e

Sigma range        : 0.10690784617684071 -> 1.0 nm
Epsilon range      : 0.0 -> 1.111270837866897 kJ/mol

Total exceptions    : 352477
R-L exceptions      : 0

=== GBn2 PARAMETERS ===
Shape : (9332, 5)
Radius range: 0.11699999999999999 -> 0.18 nm
Screen range: -0.703469 -> 1.425952

Parameter table rows: 9332

09m MM + GBn2 PARAMETERIZATION: OK


In [5]:
# ============================================================
# 09m — Build C/R/L GBn2/ACE analysis systems
# ============================================================

contexts = {}
systems = {}
integrators = {}

platform_mm = mm.Platform.getPlatformByName(PLATFORM)

components = {
    "C": S,   # complex
    "R": R,   # receptor
    "L": L,   # ligand
}

for name, indices in components.items():

    system = mm.System()

    for idx in indices:
        system.addParticle(
            physical.getParticleMass(int(idx))
        )

    # Force group 0 = GB polar only
    gb_polar = GBSAGBn2Force(
        solventDielectric=SOLVENT_DIELECTRIC,
        soluteDielectric=SOLUTE_DIELECTRIC,
        SA=None,
        cutoff=None,
        kappa=KAPPA_NM_INV
    )

    # Force group 1 = GB polar + ACE
    gb_ace = GBSAGBn2Force(
        solventDielectric=SOLVENT_DIELECTRIC,
        soluteDielectric=SOLUTE_DIELECTRIC,
        SA="ACE",
        cutoff=None,
        kappa=KAPPA_NM_INV
    )

    for idx in indices:

        particle_parameters = (
            [float(q[idx])] +
            gbparams[idx].tolist()
        )

        gb_polar.addParticle(
            particle_parameters
        )

        gb_ace.addParticle(
            particle_parameters
        )

    gb_polar.finalize()
    gb_ace.finalize()

    gb_polar.setForceGroup(0)
    gb_ace.setForceGroup(1)

    system.addForce(gb_polar)
    system.addForce(gb_ace)

    integrator = mm.VerletIntegrator(
        0.001 * unit.picosecond
    )

    properties = (
        {"Threads": str(CPU_THREADS)}
        if PLATFORM == "CPU"
        else {}
    )

    context = mm.Context(
        system,
        integrator,
        platform_mm,
        properties
    )

    systems[name] = system
    integrators[name] = integrator
    contexts[name] = context

    print(
        f"{name}: particles={system.getNumParticles()}, "
        f"forces={system.getNumForces()}"
    )

# ============================================================
# GB energy function
# ============================================================

def gb_energy(name, xyz_nm):

    ctx = contexts[name]

    ctx.setPositions(
        xyz_nm * unit.nanometer
    )

    polar = (
        ctx.getState(
            getEnergy=True,
            groups={0}
        )
        .getPotentialEnergy()
        .value_in_unit(unit.kilojoule_per_mole)
    )

    polar_plus_ace = (
        ctx.getState(
            getEnergy=True,
            groups={1}
        )
        .getPotentialEnergy()
        .value_in_unit(unit.kilojoule_per_mole)
    )

    ace = polar_plus_ace - polar

    return float(polar), float(ace)


print("\n=== GBn2/ACE SYSTEMS ===")

print("Complex  :", systems["C"].getNumParticles())
print("Receptor :", systems["R"].getNumParticles())
print("Ligand   :", systems["L"].getNumParticles())

require(
    systems["C"].getNumParticles() == 9332,
    "Complex particle count errato."
)

require(
    systems["R"].getNumParticles() == 9286,
    "Receptor particle count errato."
)

require(
    systems["L"].getNumParticles() == 46,
    "Ligand particle count errato."
)

print("\n09m GBn2/ACE SYSTEM BUILD: OK")

C: particles=9332, forces=2
R: particles=9286, forces=2
L: particles=46, forces=2

=== GBn2/ACE SYSTEMS ===
Complex  : 9332
Receptor : 9286
Ligand   : 46

09m GBn2/ACE SYSTEM BUILD: OK


In [6]:
# ============================================================
# 09m — 3-snapshot endpoint smoke test
#       MM independent validation + GBn2/ACE
# ============================================================

KJ_TO_KCAL = 1.0 / 4.184
COULOMB_CONSTANT = 138.935456  # kJ mol^-1 nm e^-2

# ------------------------------------------------------------
# Direct receptor-ligand MM interaction
# Coulomb + Lennard-Jones
# ------------------------------------------------------------

def cross_mm(xyz):
    """
    Direct R-L nonbonded interaction using the authoritative
    q, sigma and epsilon parameters.
    xyz must contain the complete solute in nm.
    """

    distances = np.linalg.norm(
        xyz[R, None, :] - xyz[L][None, :, :],
        axis=2
    )

    require(
        np.isfinite(distances).all(),
        "Distanze R/L non finite."
    )

    require(
        distances.min() > 0.05,
        "Distanza R/L <= 0.5 Å: geometria non valida."
    )

    # Lorentz-Berthelot combining rules
    sig = (
        sigma[R, None] +
        sigma[L][None, :]
    ) / 2.0

    eps = np.sqrt(
        epsilon[R, None] *
        epsilon[L][None, :]
    )

    # Coulomb
    coul = np.sum(
        COULOMB_CONSTANT *
        q[R, None] *
        q[L][None, :] /
        distances
    )

    # Lennard-Jones
    sr6 = (sig / distances) ** 6

    lj = np.sum(
        4.0 * eps * (sr6**2 - sr6)
    )

    return float(coul), float(lj)


# ------------------------------------------------------------
# Independent OpenMM MM calculation
# C - R - L
# ------------------------------------------------------------

def independent_mm(indices, xyz):
    """
    Build an independent NoCutoff NonbondedForce for a component.
    Returns its MM nonbonded energy in kJ/mol.
    """

    indices = np.asarray(indices, dtype=int)

    mapping = {
        int(old): new
        for new, old in enumerate(indices)
    }

    system = mm.System()

    force = mm.NonbondedForce()
    force.setNonbondedMethod(
        mm.NonbondedForce.NoCutoff
    )
    force.setUseDispersionCorrection(False)

    for old in indices:

        old = int(old)

        system.addParticle(
            physical.getParticleMass(old)
        )

        force.addParticle(
            q[old],
            sigma[old],
            epsilon[old]
        )

    # Preserve authoritative intramolecular exceptions
    for j in range(nbforce.getNumExceptions()):

        a, b, chargeProd, sig, eps = (
            nbforce.getExceptionParameters(j)
        )

        a = int(a)
        b = int(b)

        if a in mapping and b in mapping:

            force.addException(
                mapping[a],
                mapping[b],
                chargeProd,
                sig,
                eps
            )

    system.addForce(force)

    integ = mm.VerletIntegrator(
        0.001 * unit.picosecond
    )

    props = (
        {"Threads": str(CPU_THREADS)}
        if PLATFORM == "CPU"
        else {}
    )

    ctx = mm.Context(
        system,
        integ,
        platform_mm,
        props
    )

    component_xyz = xyz[indices]

    ctx.setPositions(
        component_xyz * unit.nanometer
    )

    energy = (
        ctx.getState(getEnergy=True)
        .getPotentialEnergy()
        .value_in_unit(unit.kilojoule_per_mole)
    )

    del ctx
    del integ

    return float(energy)


# ------------------------------------------------------------
# Independent validation on first selected snapshot
# ------------------------------------------------------------

first_frame = int(frames[0])
xyz0 = trajectory.xyz[first_frame]

direct_coul, direct_lj = cross_mm(xyz0)

mm_C = independent_mm(S, xyz0)
mm_R = independent_mm(R, xyz0)
mm_L = independent_mm(L, xyz0)

independent_delta_mm = (
    mm_C - mm_R - mm_L
)

direct_delta_mm = (
    direct_coul + direct_lj
)

mm_error = abs(
    independent_delta_mm -
    direct_delta_mm
)

print("=== INDEPENDENT MM VALIDATION ===")

print(
    f"Direct Coulomb       : "
    f"{direct_coul * KJ_TO_KCAL:.6f} kcal/mol"
)

print(
    f"Direct vdW           : "
    f"{direct_lj * KJ_TO_KCAL:.6f} kcal/mol"
)

print(
    f"Direct MM total      : "
    f"{direct_delta_mm * KJ_TO_KCAL:.6f} kcal/mol"
)

print(
    f"OpenMM C-R-L         : "
    f"{independent_delta_mm * KJ_TO_KCAL:.6f} kcal/mol"
)

print(
    f"Absolute difference  : "
    f"{mm_error:.8f} kJ/mol"
)

require(
    mm_error < 0.05,
    "Independent MM validation failed."
)

print("MM independent validation: PASS")


# ------------------------------------------------------------
# Endpoint calculation for the 3 snapshots
# ------------------------------------------------------------

rows = []

print("\n=== 09m 3-SNAPSHOT ENDPOINT TEST ===")

for count, frame_idx in enumerate(frames, start=1):

    frame_idx = int(frame_idx)

    xyz = trajectory.xyz[frame_idx]

    # Direct receptor-ligand MM
    elec, vdw = cross_mm(xyz)

    # GB polar + ACE for C, R, L
    C_polar, C_ace = gb_energy(
        "C",
        xyz[S]
    )

    R_polar, R_ace = gb_energy(
        "R",
        xyz[R]
    )

    L_polar, L_ace = gb_energy(
        "L",
        xyz[L]
    )

    delta_polar = (
        C_polar -
        R_polar -
        L_polar
    )

    delta_ace = (
        C_ace -
        R_ace -
        L_ace
    )

    endpoint = (
        elec +
        vdw +
        delta_polar +
        delta_ace
    )

    row = {
        "frame_index_0": frame_idx,
        "time_ps": (frame_idx + 1) * FRAME_PS,

        "MM_electrostatic":
            elec * KJ_TO_KCAL,

        "MM_vdw":
            vdw * KJ_TO_KCAL,

        "delta_GB_polar":
            delta_polar * KJ_TO_KCAL,

        "delta_SA_ACE":
            delta_ace * KJ_TO_KCAL,

        "endpoint_proxy":
            endpoint * KJ_TO_KCAL,

        "C_GB_polar":
            C_polar * KJ_TO_KCAL,

        "R_GB_polar":
            R_polar * KJ_TO_KCAL,

        "L_GB_polar":
            L_polar * KJ_TO_KCAL,

        "C_SA_ACE":
            C_ace * KJ_TO_KCAL,

        "R_SA_ACE":
            R_ace * KJ_TO_KCAL,

        "L_SA_ACE":
            L_ace * KJ_TO_KCAL,
    }

    require(
        np.isfinite(
            list(row.values())
        ).all(),
        f"Energie non finite al frame {frame_idx}."
    )

    rows.append(row)

    print(
        f"{count}/{len(frames)} | "
        f"{row['time_ps']:.0f} ps | "
        f"Coulomb {row['MM_electrostatic']:.2f} | "
        f"vdW {row['MM_vdw']:.2f} | "
        f"GB {row['delta_GB_polar']:.2f} | "
        f"ACE {row['delta_SA_ACE']:.2f} | "
        f"endpoint {row['endpoint_proxy']:.2f} kcal/mol"
    )


test_energies = pd.DataFrame(rows)

# ------------------------------------------------------------
# Component-sum QC
# ------------------------------------------------------------

component_sum = (
    test_energies["MM_electrostatic"] +
    test_energies["MM_vdw"] +
    test_energies["delta_GB_polar"] +
    test_energies["delta_SA_ACE"]
)

sum_ok = np.allclose(
    component_sum,
    test_energies["endpoint_proxy"],
    atol=1e-6
)

require(
    sum_ok,
    "La somma dei componenti non coincide con endpoint_proxy."
)

print("\n=== TEST RESULTS ===")

print(
    test_energies[
        [
            "time_ps",
            "MM_electrostatic",
            "MM_vdw",
            "delta_GB_polar",
            "delta_SA_ACE",
            "endpoint_proxy"
        ]
    ].to_string(index=False)
)

print(
    "\nMean endpoint:",
    f"{test_energies['endpoint_proxy'].mean():.3f}",
    "kcal/mol"
)

print(
    "SD:",
    f"{test_energies['endpoint_proxy'].std():.3f}",
    "kcal/mol"
)

print("\nComponent sum QC: PASS")
print("09m 3-SNAPSHOT SMOKE TEST: COMPLETE")

=== INDEPENDENT MM VALIDATION ===
Direct Coulomb       : -4.779664 kcal/mol
Direct vdW           : -29.704793 kcal/mol
Direct MM total      : -34.484456 kcal/mol
OpenMM C-R-L         : -34.483888 kcal/mol
Absolute difference  : 0.00237699 kJ/mol
MM independent validation: PASS

=== 09m 3-SNAPSHOT ENDPOINT TEST ===
1/3 | 202 ps | Coulomb -4.78 | vdW -29.70 | GB 23.84 | ACE -6.25 | endpoint -16.89 kcal/mol
2/3 | 600 ps | Coulomb -25.59 | vdW -28.26 | GB 41.15 | ACE -6.10 | endpoint -18.80 kcal/mol
3/3 | 1000 ps | Coulomb -2.26 | vdW -24.10 | GB 18.06 | ACE -5.28 | endpoint -13.56 kcal/mol

=== TEST RESULTS ===
 time_ps  MM_electrostatic     MM_vdw  delta_GB_polar  delta_SA_ACE  endpoint_proxy
   202.0         -4.779664 -29.704793       23.842153     -6.252466      -16.894769
   600.0        -25.590090 -28.261331       41.152391     -6.096938      -18.795967
  1000.0         -2.255500 -24.097424       18.063102     -5.275132      -13.564954

Mean endpoint: -16.419 kcal/mol
SD: 2.648 kcal/

In [7]:
# ============================================================
# 09m — OFFICIAL 50-SNAPSHOT ENDPOINT ANALYSIS
# MOL00484–EGFR
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import time
import json

N_SNAPSHOTS_PROD = 50
N_BLOCKS = 5

# ------------------------------------------------------------
# Official output directory
# ------------------------------------------------------------

OUTPUT_BASE = (
    PROJECT_ROOT
    / "outputs"
    / "pipeline_5_pharmacophore_09m_mol00484_endpoint"
)

RUN_ID = (
    datetime.now(timezone.utc)
    .strftime("%Y%m%dT%H%M%S_%fZ_production")
)

RUN_09M = OUTPUT_BASE / RUN_ID

(RUN_09M / "tables").mkdir(
    parents=True,
    exist_ok=False
)

print("=== 09m OFFICIAL ENDPOINT PRODUCTION ===")
print("RUN_ID :", RUN_ID)
print("Output :", RUN_09M)

# ------------------------------------------------------------
# 50 uniformly distributed snapshots after 20% discard
# ------------------------------------------------------------

start = int(
    trajectory.n_frames * DISCARD_FRACTION
)

frames_prod = np.linspace(
    start,
    trajectory.n_frames - 1,
    N_SNAPSHOTS_PROD,
    dtype=int
)

require(
    len(np.unique(frames_prod)) == N_SNAPSHOTS_PROD,
    "Frame duplicati nella selezione production."
)

selected_frames = pd.DataFrame({
    "frame_index_0": frames_prod,
    "time_ps": (frames_prod + 1) * FRAME_PS
})

selected_frames.to_csv(
    RUN_09M / "tables" / "selected_frames.csv",
    index=False
)

print()
print("First frame :", frames_prod[0],
      "->", (frames_prod[0] + 1) * FRAME_PS, "ps")

print("Last frame  :", frames_prod[-1],
      "->", (frames_prod[-1] + 1) * FRAME_PS, "ps")

print("Snapshots   :", len(frames_prod))

# ------------------------------------------------------------
# Provenance
# ------------------------------------------------------------

provenance = {
    "candidate": "MOL00484",
    "tier": "TIER_2_ORTHOGONAL_FOLLOWUP_LEAD",
    "source_pipeline": "09l",
    "source_run":
        "20261027T171745_698045Z_seed20261027",
    "source_MD_ps": 1000.0,
    "discard_fraction": DISCARD_FRACTION,
    "n_snapshots": N_SNAPSHOTS_PROD,
    "GB_model": "GBn2",
    "SA_model": "ACE",
    "solute_dielectric": SOLUTE_DIELECTRIC,
    "solvent_dielectric": SOLVENT_DIELECTRIC,
    "kappa_nm_inv": KAPPA_NM_INV,
    "single_trajectory": True,
    "entropy_included": False,
    "separate_relaxation": False,
    "experimentally_validated": False,
    "status": "running"
}

with open(
    RUN_09M / "provenance.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        provenance,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Endpoint calculation
# ------------------------------------------------------------

rows_prod = []

t0 = time.monotonic()

print("\n=== CALCULATION ===")

try:

    for count, frame_idx in enumerate(
        frames_prod,
        start=1
    ):

        frame_idx = int(frame_idx)

        xyz = trajectory.xyz[frame_idx]

        # Direct R-L MM
        elec, vdw = cross_mm(xyz)

        # GBn2/ACE C/R/L
        C_polar, C_ace = gb_energy(
            "C",
            xyz[S]
        )

        R_polar, R_ace = gb_energy(
            "R",
            xyz[R]
        )

        L_polar, L_ace = gb_energy(
            "L",
            xyz[L]
        )

        delta_polar = (
            C_polar -
            R_polar -
            L_polar
        )

        delta_ace = (
            C_ace -
            R_ace -
            L_ace
        )

        endpoint = (
            elec +
            vdw +
            delta_polar +
            delta_ace
        )

        row = {
            "frame_index_0": frame_idx,
            "time_ps":
                (frame_idx + 1) * FRAME_PS,

            "MM_electrostatic":
                elec * KJ_TO_KCAL,

            "MM_vdw":
                vdw * KJ_TO_KCAL,

            "delta_GB_polar":
                delta_polar * KJ_TO_KCAL,

            "delta_SA_ACE":
                delta_ace * KJ_TO_KCAL,

            "endpoint_proxy":
                endpoint * KJ_TO_KCAL,

            "C_GB_polar":
                C_polar * KJ_TO_KCAL,

            "R_GB_polar":
                R_polar * KJ_TO_KCAL,

            "L_GB_polar":
                L_polar * KJ_TO_KCAL,

            "C_SA_ACE":
                C_ace * KJ_TO_KCAL,

            "R_SA_ACE":
                R_ace * KJ_TO_KCAL,

            "L_SA_ACE":
                L_ace * KJ_TO_KCAL,
        }

        require(
            np.isfinite(
                list(row.values())
            ).all(),
            f"Energie non finite al frame {frame_idx}."
        )

        rows_prod.append(row)

        # Save immediately
        pd.DataFrame(
            rows_prod
        ).to_csv(
            RUN_09M
            / "tables"
            / "endpoint_per_frame_kcal_mol.csv",
            index=False
        )

        elapsed = time.monotonic() - t0

        remaining = (
            elapsed / count
            * (N_SNAPSHOTS_PROD - count)
        )

        print(
            f"{count:02d}/{N_SNAPSHOTS_PROD} "
            f"({100*count/N_SNAPSHOTS_PROD:5.1f}%) | "
            f"{row['time_ps']:6.0f} ps | "
            f"endpoint "
            f"{row['endpoint_proxy']:8.2f} kcal/mol | "
            f"ETA {remaining/60:5.1f} min",
            flush=True
        )

except Exception as exc:

    provenance["status"] = "failed"
    provenance["error"] = repr(exc)
    provenance["completed_snapshots"] = len(
        rows_prod
    )

    with open(
        RUN_09M / "provenance.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            provenance,
            f,
            indent=2
        )

    raise

energies = pd.DataFrame(rows_prod)

# ------------------------------------------------------------
# QC: component sum
# ------------------------------------------------------------

component_sum = (
    energies["MM_electrostatic"]
    + energies["MM_vdw"]
    + energies["delta_GB_polar"]
    + energies["delta_SA_ACE"]
)

sum_ok = np.allclose(
    component_sum,
    energies["endpoint_proxy"],
    atol=1e-6
)

require(
    sum_ok,
    "Component-sum QC failed."
)

require(
    len(energies) == 50,
    "Non sono stati completati tutti i 50 snapshot."
)

# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

columns = [
    "MM_electrostatic",
    "MM_vdw",
    "delta_GB_polar",
    "delta_SA_ACE",
    "endpoint_proxy"
]

stats = energies[columns].agg(
    ["mean", "std", "min", "max"]
).T

stats.to_csv(
    RUN_09M
    / "tables"
    / "endpoint_summary_statistics.csv"
)

# ------------------------------------------------------------
# Temporal blocks
# ------------------------------------------------------------

block_rows = []

for k, idx in enumerate(
    np.array_split(
        np.arange(len(energies)),
        N_BLOCKS
    ),
    start=1
):

    block_rows.append({
        "block": k,
        "time_start_ps":
            energies.iloc[idx]["time_ps"].min(),

        "time_end_ps":
            energies.iloc[idx]["time_ps"].max(),

        "n": len(idx),

        "endpoint_mean_kcal_mol":
            energies.iloc[idx][
                "endpoint_proxy"
            ].mean()
    })

blocks = pd.DataFrame(block_rows)

blocks.to_csv(
    RUN_09M
    / "tables"
    / "temporal_blocks.csv",
    index=False
)

# ------------------------------------------------------------
# First/second half
# ------------------------------------------------------------

mid = len(energies) // 2

first_half = float(
    energies.iloc[:mid][
        "endpoint_proxy"
    ].mean()
)

second_half = float(
    energies.iloc[mid:][
        "endpoint_proxy"
    ].mean()
)

# ------------------------------------------------------------
# Final QC
# ------------------------------------------------------------

qc = {
    "all_requested_snapshots":
        len(energies) == 50,

    "finite_energies":
        bool(
            np.isfinite(
                energies[columns]
            ).all().all()
        ),

    "MM_independent_validation":
        mm_error < 0.05,

    "sum_components_matches":
        bool(sum_ok)
}

pd.DataFrame([
    {
        "check": k,
        "pass": v
    }
    for k, v in qc.items()
]).to_csv(
    RUN_09M
    / "tables"
    / "09m_QC.csv",
    index=False
)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

result = {
    "candidate": "MOL00484",
    "tier":
        "TIER_2_ORTHOGONAL_FOLLOWUP_LEAD",

    "mode": "production",

    "n_snapshots":
        len(energies),

    "source_MD_ps":
        1000.0,

    "GB_model":
        "GBn2",

    "SA_model":
        "ACE",

    "mean_endpoint_kcal_mol":
        float(
            energies[
                "endpoint_proxy"
            ].mean()
        ),

    "SD_across_correlated_snapshots_kcal_mol":
        float(
            energies[
                "endpoint_proxy"
            ].std()
        ),

    "first_half_mean_kcal_mol":
        first_half,

    "second_half_mean_kcal_mol":
        second_half,

    "technical_qc_pass":
        bool(all(qc.values())),

    "interpretation":
        (
            "Single-trajectory MM/GBSA-like "
            "endpoint proxy; no entropy; "
            "not rigorous binding free energy."
        )
}

with open(
    RUN_09M / "09m_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        result,
        f,
        indent=2
    )

provenance["status"] = "completed"
provenance["result"] = result

with open(
    RUN_09M / "provenance.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        provenance,
        f,
        indent=2
    )

# ------------------------------------------------------------
# Print final results
# ------------------------------------------------------------

print("\n=== 09m OFFICIAL RESULTS ===")

print(
    "\nMean components "
    "(kcal/mol):"
)

for col in columns:
    print(
        f"{col:20s}: "
        f"{energies[col].mean():8.3f} "
        f"± {energies[col].std():6.3f}"
    )

print(
    "\nFirst half mean :",
    f"{first_half:.3f} kcal/mol"
)

print(
    "Second half mean:",
    f"{second_half:.3f} kcal/mol"
)

print("\nTemporal blocks:")
print(
    blocks.to_string(index=False)
)

print(
    "\nTechnical QC:",
    all(qc.values())
)

print(
    "Output:",
    RUN_09M
)

print(
    "\n09m OFFICIAL 50-SNAPSHOT "
    "ENDPOINT ANALYSIS: COMPLETE"
)

=== 09m OFFICIAL ENDPOINT PRODUCTION ===
RUN_ID : 20261028T062629_605626Z_production
Output : C:\Users\Roberto\PEARL\outputs\pipeline_5_pharmacophore_09m_mol00484_endpoint\20261028T062629_605626Z_production

First frame : 100 -> 202.0 ps
Last frame  : 499 -> 1000.0 ps
Snapshots   : 50

=== CALCULATION ===
01/50 (  2.0%) |    202 ps | endpoint   -16.89 kcal/mol | ETA  12.1 min
02/50 (  4.0%) |    218 ps | endpoint   -13.59 kcal/mol | ETA  12.0 min
03/50 (  6.0%) |    234 ps | endpoint   -14.68 kcal/mol | ETA  11.7 min
04/50 (  8.0%) |    250 ps | endpoint   -15.32 kcal/mol | ETA  11.4 min
05/50 ( 10.0%) |    266 ps | endpoint   -14.08 kcal/mol | ETA  11.0 min
06/50 ( 12.0%) |    282 ps | endpoint   -14.21 kcal/mol | ETA  10.8 min
07/50 ( 14.0%) |    298 ps | endpoint   -16.82 kcal/mol | ETA  10.6 min
08/50 ( 16.0%) |    316 ps | endpoint   -15.60 kcal/mol | ETA  10.3 min
09/50 ( 18.0%) |    332 ps | endpoint   -14.90 kcal/mol | ETA  10.1 min
10/50 ( 20.0%) |    348 ps | endpoint   -17.1

In [1]:
from pathlib import Path

BASE = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_pharmacophore_09m_mol00484_endpoint"
)

print("BASE exists:", BASE.exists())
print("BASE:", BASE)

if BASE.exists():

    runs = sorted(
        [p for p in BASE.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("\nRuns found:", len(runs))

    for i, run in enumerate(runs):
        print(f"\n[{i}] {run.name}")

        tables = run / "tables"

        if tables.exists():
            print("    TABLES:")
            for f in sorted(tables.iterdir()):
                if f.is_file():
                    print(
                        f"      {f.name:40s} "
                        f"{f.stat().st_size / 1024:.1f} KB"
                    )

        plots = run / "plots"

        if plots.exists():
            print("    PLOTS:")
            for f in sorted(plots.iterdir()):
                if f.is_file():
                    print(
                        f"      {f.name:40s} "
                        f"{f.stat().st_size / 1024:.1f} KB"
                    )

BASE exists: True
BASE: C:\Users\Roberto\PEARL\outputs\pipeline_5_pharmacophore_09m_mol00484_endpoint

Runs found: 1

[0] 20261028T062629_605626Z_production
    TABLES:
      09m_QC.csv                               0.1 KB
      endpoint_per_frame_kcal_mol.csv          10.9 KB
      endpoint_summary_statistics.csv          0.5 KB
      selected_frames.csv                      0.6 KB
      temporal_blocks.csv                      0.2 KB


In [2]:
from pathlib import Path
import pandas as pd

RUN = Path(
    r"C:\Users\Roberto\PEARL\outputs"
    r"\pipeline_5_pharmacophore_09m_mol00484_endpoint"
    r"\20261028T062629_605626Z_production"
)

TABLES = RUN / "tables"

files = [
    "endpoint_per_frame_kcal_mol.csv",
    "endpoint_summary_statistics.csv",
    "temporal_blocks.csv",
    "selected_frames.csv",
    "09m_QC.csv",
]

for name in files:

    path = TABLES / name

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    print("\nContent:")
    print(df.to_string(index=False))


endpoint_per_frame_kcal_mol.csv
Shape: (50, 13)
Columns:
['frame_index_0', 'time_ps', 'MM_electrostatic', 'MM_vdw', 'delta_GB_polar', 'delta_SA_ACE', 'endpoint_proxy', 'C_GB_polar', 'R_GB_polar', 'L_GB_polar', 'C_SA_ACE', 'R_SA_ACE', 'L_SA_ACE']

Content:
 frame_index_0  time_ps  MM_electrostatic     MM_vdw  delta_GB_polar  delta_SA_ACE  endpoint_proxy   C_GB_polar   R_GB_polar  L_GB_polar   C_SA_ACE   R_SA_ACE  L_SA_ACE
           100    202.0         -4.779664 -29.704793       23.841225     -6.249489      -16.892720 -7483.095237 -7480.139495  -26.796966 333.330829 333.035861  6.544456
           108    218.0          5.522131 -31.002876       18.333905     -6.443518      -13.590358 -7489.694903 -7480.580763  -27.448046 324.581531 324.828508  6.196541
           116    234.0          3.250820 -29.998890       18.569369     -6.503398      -14.682099 -7461.258385 -7451.041388  -28.786366 324.527862 324.379080  6.652179
           124    250.0          2.810226 -32.665472       20.58033